# Squares in Squares — the live frontier atlas
This page tracks the **public leaderboard frontier**: for every n, the shortest side that any public notebook output currently reaches. It is rebuilt several times a day from the outputs of public notebooks, and shows **who moved the frontier, which n changed, how the packings looked before and after, and where empty space remains**. Every picture comes from a public output and is credited to its author. Nothing on this page is a claim of a new packing.

> **Copy & Edit:** this notebook writes **no** submission file by default (`WRITE_SUBMISSION = False`). The `submission.csv` in the attached dataset is the current best public output, copied unchanged and credited in section 10.

In [ ]:
WRITE_SUBMISSION = False   # Copy & Edit: set True and Run All to copy the frontier file to submission.csv
import glob, os, json, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle, Rectangle
from scipy import ndimage
from scipy.spatial import cKDTree
from IPython.display import display, Markdown

INK, MUTED, GRID = '#0f172a', '#64748b', '#e2e8f0'
BLUE, ORANGE, GREEN, RED, LBLUE = '#2563eb', '#ea580c', '#16a34a', '#dc2626', '#bfdbfe'
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'axes.edgecolor': GRID, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': .6, 'font.size': 10, 'axes.titleweight': 'bold', 'axes.titlesize': 11,
    'axes.titlelocation': 'left'})

def find(name):
    c = glob.glob(f'/kaggle/input/**/atlas_{name}', recursive=True) or [os.path.expanduser(f'~/kaggle_squares/atlas/out/{name}')]
    return c[0]
fr = pd.read_csv(find('frontier.csv')); src = pd.read_csv(find('sources.csv')); hist = pd.read_csv(find('history.csv'))
fhist = pd.read_csv(find('frontier_history.csv')); clog = pd.read_csv(find('changes_log.csv')); show = json.load(open(find('showcase.json')))
N = fr.n.to_numpy(); sqn = np.sqrt(N)
score = lambda L: float((np.asarray(L)/sqn-1).sum())
F = score(fr.L_frontier); built = show['built_utc']
who = lambda ref: (src.set_index('ref').author.get(ref) if ref in set(src.ref) else None) or str(ref).split('/')[0]
print(f'data built {built} UTC | frontier {F:.10f} | public outputs tracked {len(src)} | changes logged {len(clog)}')

In [ ]:
# KPI tiles
last = fhist.iloc[-1]; prev = fhist.iloc[-2] if len(fhist) > 1 else None
leader = src.iloc[0]
fig, axs = plt.subplots(1, 4, figsize=(11, 1.9))
tiles = [('FRONTIER SCORE', f'{F:.6f}', GREEN, 'lower is better'), ('LEADING OUTPUT', who(leader.ref)[:16], INK, leader.title[:34]),
         ('LAST MOVE', f'-{prev.frontier-last.frontier:.6f}' if prev is not None else 'first snapshot', BLUE, f'at {last.seen_at} UTC'),
         ('PUBLIC OUTPUTS TRACKED', f'{len(src)}', INK, f'refreshed {built} UTC')]
for ax, (t, v, c, s) in zip(axs, tiles):
    ax.axis('off'); ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fc='#f8fafc', ec=GRID, lw=1))
    ax.text(.06, .76, t, transform=ax.transAxes, fontsize=8, color=MUTED, weight='bold')
    ax.text(.06, .38, v, transform=ax.transAxes, fontsize=17 if len(v) < 14 else 13, color=c, weight='bold'); ax.text(.06, .12, s, transform=ax.transAxes, fontsize=8, color=MUTED)
plt.tight_layout(); plt.show()

## 1. Who is where
Each bar is one public output (lower total score is better). The differences between the top outputs sit in the 5th–8th decimal, so the axis shows the **gap to the frontier** on a log scale.

In [ ]:
s = src.copy(); s['gap'] = s.score - F; s = s.head(10).iloc[::-1]
lab = [(a if isinstance(a, str) else '')[:22] or r.split('/')[0] for a, r in zip(s.author, s.ref)]
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.barh(lab, s.gap.clip(lower=1e-9), color=[GREEN if g < 1e-7 else '#94a3b8' for g in s.gap]); ax.set_xscale('log'); ax.set_xlabel('gap to the frontier (score units, log scale, floor 1e-9)')
for y, (g, sc_) in enumerate(zip(s.gap, s.score)): ax.text(max(g, 1e-9)*1.2, y, f'{sc_:.6f}', va='center', fontsize=8, color=MUTED)
ax.set_title('Public outputs vs the frontier'); ax.grid(axis='y', visible=False); plt.tight_layout(); plt.show()

## 2. How the public best moved
Points before the first automatic refresh (◇) are **backfilled by hand** from public notebook titles; from then on every step is recorded by the collector.

In [ ]:
manual = pd.DataFrame([('2026-11-15 12:00', 22.148794, 'jazivxt: Boxed In'), ('2026-11-15 20:00', 22.146664, 'lucifer19: Think. Select. Reclose'),
    ('2026-11-17 02:00', 22.137005, 'jazivxt: Boxed In Sub'), ('2026-11-20 22:00', 22.107668, 'Alex Chilton: Starter Kit'),
    ('2026-11-21 00:00', 22.107642, 'Areeb Ali: certified micro-shrink'), ('2026-11-21 02:00', 22.107615, 'prvsiyan: A Square Deal'),
    ('2026-11-23 11:00', 22.092662, 'jonathan1506 / mizero')], columns=['t', 'score', 'who'])
manual['t'] = pd.to_datetime(manual.t); auto = fhist.copy(); auto['t'] = pd.to_datetime(auto.seen_at)
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.step(manual.t, manual.score.cummin(), where='post', color=GREEN, lw=2); ax.scatter(manual.t, manual.score, marker='D', s=26, color=GREEN, zorder=3)
for i, (_, r) in enumerate(manual.iterrows()):
    if r.who.startswith('Areeb'): continue
    ax.annotate(r.who, (r.t, r.score), xytext=(6, 6+(i % 2)*10), textcoords='offset points', fontsize=7, color=MUTED, ha='left' if i < 5 else 'right')
if len(auto): ax.plot(auto.t, auto.frontier, 'o-', color=GREEN, ms=4, label='recorded by the collector')
ax.set_ylabel('best public score'); ax.set_ylim(F-.004, 22.155); ax.set_title('Public frontier over time (lower is better)')
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 3. Which n belongs to whom
For every n the colour shows the public output that reaches the shortest side (a tie goes to the notebook that ran first). A mix of colours means the frontier is being shared between authors; one colour means a single merged file leads everywhere.

In [ ]:
hold = fr.holder.fillna('?'); names = list(hold.value_counts().index)
pal = [GREEN, BLUE, ORANGE, '#7c3aed', '#0891b2', '#be123c', '#a16207', '#475569', '#db2777', '#65a30d']
cmap = {h: pal[i] if i < len(pal) else '#cbd5e1' for i, h in enumerate(names)}
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 3.9), gridspec_kw={'height_ratios': [1, 1.1]})
ax.bar(fr.n, 1, width=1, color=[cmap[h] for h in hold]); ax.set_yticks([]); ax.set_xlim(1, 1000); ax.grid(False); ax.set_title('Holder of the shortest side, n = 1 … 1000')
cnt = hold.value_counts().head(8).iloc[::-1]
ax2.barh([who(c)[:24] for c in cnt.index], cnt.values, color=[cmap[c] for c in cnt.index]); ax2.set_xlabel('number of n held'); ax2.grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

## 4. What changed
Every time the collector sees the frontier improve at some n, the change is logged. The first line of this section is the latest refresh; the chart is the score gained per n; the table is the log.

In [ ]:
c = clog.copy(); c['seen_at'] = pd.to_datetime(c.seen_at)
lastt = c.seen_at.max(); cur = c[c.seen_at == lastt]
note = ' (first snapshot: compared with the previous public tier)' if cur.seeded.astype(str).str.lower().eq('true').all() else ''
display(Markdown(f"**Latest refresh {lastt:%d %b %H:%M} UTC:** {len(cur)} n improved, total score gained **{cur.gain.sum():.6f}**{note}."))
fig, ax = plt.subplots(figsize=(10, 3.2)); ax.bar(cur.n, cur.gain, width=1.2, color=GREEN); ax.set_xlim(1, 1000)
ax.set_xlabel('n'); ax.set_ylabel('score gained at n'); ax.set_title('Where the latest improvement landed')
for k in range(2, 32): ax.axvline(k*k, color=GRID, lw=.5, zorder=0)
plt.tight_layout(); plt.show()
t = c.sort_values(['seen_at', 'gain'], ascending=[False, False]).head(12).copy()
t['seen_at'] = t.seen_at.dt.strftime('%d %b %H:%M'); t['gain'] = t.gain.map('{:.2e}'.format); t['from'] = t.old_holder.map(who); t['to'] = t.new_holder.map(who)
display(t[['seen_at', 'n', 'gain', 'from', 'to']].reset_index(drop=True))

### Who moves the frontier
Score gained at each refresh, credited to the output that holds the n after the change. The first block is the first snapshot, measured against the previous public tier, so it is not one day of work.

In [ ]:
c['who'] = c.new_holder.map(who); c['day'] = c.seen_at.dt.strftime('%d %b')
tot = c.groupby('who').gain.sum().sort_values().tail(10)
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={'width_ratios': [1.1, 1]})
ax.barh([w[:22] for w in tot.index], tot.values, color=BLUE); ax.set_xlabel('score gained (all logged changes)'); ax.set_title('Credit for the frontier'); ax.grid(axis='y', visible=False)
piv = c.pivot_table(index='day', columns='who', values='gain', aggfunc='sum', fill_value=0, sort=False)
bottom = np.zeros(len(piv))
for j, col in enumerate(piv.columns): ax2.bar(piv.index, piv[col], bottom=bottom, color=pal[j % len(pal)] if 'pal' in globals() else BLUE, label=col[:18]); bottom += piv[col].to_numpy()
ax2.set_title('Score gained per day'); ax2.legend(frameon=False, fontsize=7, ncol=1, loc='upper right'); plt.setp(ax2.get_xticklabels(), rotation=30)
plt.tight_layout(); plt.show()

## 5. The latest changes, as pictures
Left: the **packing before** the change (previous frontier holder). Right: the **packing after** (current holder). Orange squares are tilted, blue ones are axis-aligned. A red outline marks squares of the new packing with no counterpart (same place and angle) in the old one; when the two layouts are completely different no outline is drawn.

In [ ]:
def draw(ax, geo, hl=None, title=''):
    a = np.array(geo, float); x, y, t = a.T; h = .5*(np.abs(np.cos(t))+np.abs(np.sin(t)))
    x0, y0 = (x-h).min(), (y-h).min(); L = max((x+h).max()-x0, (y+h).max()-y0)
    til = np.minimum(t % (np.pi/2), np.pi/2-t % (np.pi/2)) > 1e-4
    lw = .3 if len(a) < 400 else .15
    for i, (xx, yy, tt) in enumerate(a):
        c, s_ = np.cos(tt), np.sin(tt)
        pts = [(xx-x0+c*u-s_*v, yy-y0+s_*u+c*v) for u, v in ((-.5, -.5), (.5, -.5), (.5, .5), (-.5, .5))]
        red = hl is not None and hl[i]
        ax.add_patch(Polygon(pts, closed=True, fc=ORANGE if til[i] else LBLUE, ec=RED if red else '#1e293b', lw=1.1 if red else lw, alpha=.95, zorder=3 if red else 2))
    ax.set_xlim(0, L); ax.set_ylim(0, L); ax.set_aspect('equal'); ax.axis('off'); ax.set_title(title, fontsize=9)
def norm(geo):
    a = np.array(geo, float); h = .5*(np.abs(np.cos(a[:, 2]))+np.abs(np.sin(a[:, 2]))); a[:, 0] -= (a[:, 0]-h).min(); a[:, 1] -= (a[:, 1]-h).min(); return a
def sym(a, k, L):
    a = a.copy()
    if k & 1: a[:, 0] = L-a[:, 0]; a[:, 2] = -a[:, 2]
    if k & 2: a[:, 1] = L-a[:, 1]; a[:, 2] = -a[:, 2]
    if k & 4: a[:, [0, 1]] = a[:, [1, 0]]; a[:, 2] = -a[:, 2]
    return a
def changed_mask(new, old):
    L = max(new[:, 0].max(), new[:, 1].max())+.5; tr = cKDTree(old[:, :2]); best = None
    for k in range(8):
        b = sym(new, k, L); d, j = tr.query(b[:, :2]); da = np.abs((b[:, 2]-old[j, 2]+np.pi/4) % (np.pi/2)-np.pi/4)
        m = (d > 5e-3) | (da > 5e-3)
        if best is None or m.sum() < best[0].sum(): best = (m, k)
    return best[0], best[1]
ch = [s_ for s_ in show['items'] if s_['kind'] == 'changed'][:6]
for s_ in ch:
    fig, axs = plt.subplots(1, 2, figsize=(9, 4.6))
    mk, kk = changed_mask(norm(s_['after']), norm(s_['before'])); same = mk.mean() < .5
    draw(axs[0], s_['before'], title=f"before: {who(s_['holder_before'])[:24]}   L={s_['L_before']:.5f}")
    draw(axs[1], s_['after'], hl=mk if same else None, title=f"after: {who(s_['holder_after'])[:24]}   L={s_['L_after']:.5f}   (" + (f"{mk.sum()} squares differ)" if same else "a different layout)"))
    fig.suptitle(f"n = {s_['n']}   side shortened by {s_['L_before']-s_['L_after']:.5f}   ·   score gained {(s_['L_before']-s_['L_after'])/math.sqrt(s_['n']):.2e}", x=.02, ha='left', fontweight='bold')
    plt.tight_layout(); plt.show()

## 6. How the frontier packings are built
Every packing is a grid of axis-aligned squares (blue) with some squares tilted (orange). **Kind** says how: *grid* (no tilt), *pure 45°* (a rotated block, an exact construction), *one tilt family* (up to three angles, a slanted band) or *multi-tilt* (many angles, relaxed by an optimiser). The first chart counts the kinds per band k = ⌈√n⌉; the second shows how many squares are tilted at every n.

In [ ]:
kcol = {'grid': '#cbd5e1', 'pure 45 deg': GREEN, 'one tilt family': BLUE, 'multi-tilt': ORANGE}
fr['k'] = np.ceil(np.sqrt(fr.n)).astype(int)
ct = fr.pivot_table(index='k', columns='kind', values='n', aggfunc='count', fill_value=0).reindex(columns=list(kcol), fill_value=0)
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 5.6))
bot = np.zeros(len(ct))
for kk, col in kcol.items(): ax.bar(ct.index, ct[kk], bottom=bot, color=col, label=kk); bot += ct[kk].to_numpy()
ax.set_xlabel('band k = ceil(sqrt n)'); ax.set_ylabel('number of n'); ax.legend(frameon=False, ncol=4, fontsize=8, loc='upper left'); ax.set_title('Kind of the frontier packing, per band')
ax2.scatter(fr.n, fr.n_tilted, s=5, c=[kcol[k_] for k_ in fr.kind]); ax2.set_xlabel('n'); ax2.set_ylabel('tilted squares'); ax2.set_title('How many squares are tilted')
plt.tight_layout(); plt.show()
ties = fr[fr.n_tied >= 2]
display(Markdown(f"**{len(ties)} of 1000 n** are reached to within 1e-9 by two or more public outputs ({(fr.n_tied >= 3).sum()} by three or more): independent routes to the same value, usually a sign of a hard floor for current methods."))

### What a change does to the tilts
For each picture above: how many squares are tilted before and after, and which tilt angles (degrees, rounded to 0.5) the packing uses.

In [ ]:
def tinfo(geo):
    t = np.mod(np.array(geo, float)[:, 2], np.pi/2); t = np.minimum(t, np.pi/2-t); til = t > 1e-4
    ang = np.unique(np.round(np.degrees(t[til])/.5)*.5); return int(til.sum()), ang
rows = []
for s_ in [x for x in show['items'] if x['kind'] == 'changed']:
    b, ab = tinfo(s_['before']); a_, aa = tinfo(s_['after'])
    rows.append({'n': s_['n'], 'tilted before': b, 'tilted after': a_, 'angles before': ', '.join(f'{v:g}' for v in ab[:6]) + (' ...' if len(ab) > 6 else ''), 'angles after': ', '.join(f'{v:g}' for v in aa[:6]) + (' ...' if len(aa) > 6 else '')})
display(pd.DataFrame(rows))

## 7. Where there is room — and where there is not
A packing that leaves empty space is *not* automatically improvable: the space is usually thin slivers between jammed squares. The maps show the container in white where squares are, and colour every free point by its **distance to the nearest square** (purple = a hairline, yellow = open). The cyan circle is the **largest empty disc**. A unit square needs a disc of radius ≥ 0.5; below ~0.3 it is a sliver. Long thin strips along a wall are real slack, but only a re-arrangement of whole blocks could use them.

In [ ]:
def free_map(geo, res=0.02):
    a = np.array(geo, float); x, y, t = a.T; h = .5*(np.abs(np.cos(t))+np.abs(np.sin(t))); x0, y0 = (x-h).min(), (y-h).min()
    L = max((x+h).max()-x0, (y+h).max()-y0); m = int(math.ceil(L/res)); cov = np.zeros((m, m), bool)
    for xx, yy, tt in a:
        c, s_ = math.cos(tt), math.sin(tt); hh = .5*(abs(c)+abs(s_)); cx, cy = xx-x0, yy-y0
        i0, i1 = max(int((cx-hh)/res)-1, 0), min(int((cx+hh)/res)+2, m); j0, j1 = max(int((cy-hh)/res)-1, 0), min(int((cy+hh)/res)+2, m)
        gx, gy = np.meshgrid((np.arange(i0, i1)+.5)*res-cx, (np.arange(j0, j1)+.5)*res-cy, indexing='ij')
        u, v = gx*c+gy*s_, -gx*s_+gy*c; cov[i0:i1, j0:j1] |= (np.abs(u) <= .5) & (np.abs(v) <= .5)
    pad = np.pad(cov, 1, constant_values=True); d = ndimage.distance_transform_edt(~pad)[1:-1, 1:-1]*res
    return d, L, 1-cov.mean()
rows = []; sel = show['items'][:9]
fig, axs = plt.subplots(3, 3, figsize=(10.5, 10.5))
for ax, s_ in zip(axs.ravel(), sel):
    d, L, vf = free_map(s_['after']); i, j = np.unravel_index(d.argmax(), d.shape); r = d.max()
    ax.imshow(np.ma.masked_where(d <= 0, np.minimum(d, .3)).T, origin='lower', extent=(0, L, 0, L), cmap='plasma', vmin=0, vmax=.3)
    ax.add_patch(Circle(((i+.5)*.02, (j+.5)*.02), r, fill=False, ec='#22d3ee', lw=1.4)); ax.axis('off')
    ax.set_title(f"n={s_['n']}  hole r={r:.3f}  void {vf*100:.2f}%" + ('  (perfect grid)' if vf == 0 else ''), fontsize=9); rows.append((s_['n'], r, vf, L))
plt.tight_layout(); plt.show()
rr = pd.DataFrame(rows, columns=['n', 'largest_empty_disc_r', 'void_fraction', 'L'])
display(Markdown(f"Largest empty disc in these {len(rr)} frontier packings: **{rr.largest_empty_disc_r.max():.3f}** (a unit square needs 0.5). "
                 "So the free area is spread in slivers: *room exists, but not as a hole you can drop a square into.*"))

## 8. The score per n and the plateaus
`score(n) = L/√n − 1`. At the top of every band the best packing is simply the k×k grid with squares removed (L = k). Those *plateaus* carry about a third of the total, and the void area `L² − n` is largest there: the biggest visible slack of the whole table, and the hardest to turn into a gain.

In [ ]:
k = np.ceil(sqn).astype(int); sc_n = fr.L_frontier/sqn-1; plateau = np.abs(fr.L_frontier-k) < 1e-9; slack = fr.L_frontier**2-N
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 5.4), sharex=True)
ax.plot(N, sc_n, color='#94a3b8', lw=.8); ax.scatter(N[plateau], sc_n[plateau], s=4, color=RED, label=f'plateau (L = k): {plateau.sum()} n, {sc_n[plateau].sum()/sc_n.sum()*100:.0f}% of score')
ax.set_xlim(20, 1000); ax.set_ylim(-.002, .09); ax.set_ylabel('score(n)'); ax.legend(frameon=False, loc='upper right'); ax.set_title('Frontier score per n')
ax2.fill_between(N, slack, color=ORANGE, alpha=.8, lw=0); ax2.set_ylabel('void area  L² − n'); ax2.set_xlabel('n'); ax2.set_title('Empty area (in unit squares)')
plt.tight_layout(); plt.show()

## 9. Which n look least polished
A heuristic, not a proof. For every n, compare `k * score(n)` with the median of the same relative position `(n - (k-1)²) / (2k-1)` in the neighbouring bands (k ± 1…3). An n far above its neighbours is one where the frontier is worse than the pattern around it, so a better packing is plausible there. Plateaus and the first bands are excluded.

In [ ]:
kk = np.ceil(sqn).astype(int); rel = np.round((N-(kk-1)**2)/(2*kk-1), 2); v = (fr.L_frontier/sqn-1)*kk
d = pd.DataFrame({'n': N, 'k': kk, 'rel': rel, 'v': v, 'plateau': plateau})
res = []
for i, r in d.iterrows():
    if r.k < 8 or r.plateau: res.append(np.nan); continue
    nb = d[(d.k != r.k) & (abs(d.k-r.k) <= 3) & (abs(d.rel-r.rel) <= .01) & (~d.plateau)]
    res.append(r.v-nb.v.median() if len(nb) >= 3 else np.nan)
d['excess'] = res; top = d.dropna().sort_values('excess', ascending=False).head(12)
fig, ax = plt.subplots(figsize=(10, 3)); ax.scatter(d.n, d.excess, s=4, color='#94a3b8'); ax.scatter(top.n, top.excess, s=18, color=RED)
for _, r in top.iterrows(): ax.annotate(int(r.n), (r.n, r.excess), xytext=(3, 3), textcoords='offset points', fontsize=7, color=MUTED)
ax.axhline(0, color=GRID); ax.set_xlabel('n'); ax.set_ylabel('k*score vs neighbours'); ax.set_title('Distance above the pattern of neighbouring bands'); plt.tight_layout(); plt.show()
display(top[['n', 'k', 'excess']].assign(holder=lambda t: t.n.map(fr.set_index('n').holder).map(who)).round({'excess': 4}).reset_index(drop=True))

## 10. Credit and method
* All packings come from public notebook outputs; each author is named in the tables and picture titles. This notebook only compares them.
* Side lengths are measured in float64 from the rotated squares and match the official metric to about 1e-12. This page does **not** certify overlaps; the official metric does.
* Outputs whose score looks like an exploit (below 22) are ignored.
* The collector keeps the previous frontier as a snapshot, so each refresh shows exactly what moved since the last one.

In [ ]:
sj = json.load(open(find('submission.json')))
display(Markdown(f"**`submission.csv` in the attached dataset** is the output of the public notebook \"{sj['title']}\" by {sj['author']} ({sj['ref']}), copied byte for byte (SHA-256 {sj['sha256'][:16]}…), score {sj['score']:.10f}. No packing is changed here."))
t = src.head(12)[['author', 'title', 'score', 'votes']].copy(); t['score'] = t.score.map('{:.9f}'.format)
display(t.reset_index(drop=True)); print('Data refreshed (UTC):', built)

In [ ]:
if WRITE_SUBMISSION:
    import shutil; p = '/kaggle/input/squares-in-squares-best/submission.csv'; p = p if os.path.exists(p) else glob.glob('/kaggle/input/**/submission.csv', recursive=True)[0]
    shutil.copy(p, 'submission.csv'); print('written submission.csv from', p, '(best public output, unchanged)')
else:
    print('WRITE_SUBMISSION is False: no submission file is written.')